## This script is an example of kitti evaluation

In [1]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import shutil
from iwod.model.lightning_module import LitPSDepth
from iwod.dataset import kitti_multiview_detection
import lightning as L
import torchvision
from iwod.utils.transforms import *
from torch.utils.data import DataLoader
import iwod.eval_kitti.kitti_common as kitti_eval
from iwod.eval_kitti.eval import get_official_eval_result
import copy
from iwod.model.submodules import *
import detectron2.layers as dt2
import yaml
from iwod.eval_kitti.utils import load_config, read_imageset_file, get_prediction_dict, convert_targets

User input is required for DATASET_PATH, TESTFILE, CONFIG_PATH, and CHECKPOINT_PATH

In [2]:
#### User configurations
DATASET_PATH = "/mnt/deepdoubt/dennis_data/mulitview_kitti"
TESTFILE = "/home/dennis/git_repos/IWOD/config/kitti_config/val_test.txt"
CONFIG_PATH = "/home/dennis/git_repos/IWOD/config/train_cam2_cam0.yaml"
CHECKPOINT_PATH = "/mnt/deepdoubt/dennis_data/logging/kitti_stereo/2001/kitti-stereo-epoch=104-train_loss=0.00.ckpt"
####


LABELPATH = os.path.join(DATASET_PATH, "data_object_label_2/training/label_2")
CALIBPATH = os.path.join(DATASET_PATH, "data_object_calib/training/calib")
TARGETPATH = os.path.join(DATASET_PATH, "eval_targets")
DEVICE = 2

Prepare targets

In [3]:
convert_targets(TESTFILE, LABELPATH, CALIBPATH, TARGETPATH)

LOAD Model

In [4]:
# folder to load config file
cfg = load_config(CONFIG_PATH)
model = LitPSDepth.load_from_checkpoint(CHECKPOINT_PATH, cfg=cfg)
model.eval()
trainer = L.Trainer(devices=[DEVICE], precision="32-true")

/home/dennis/miniconda3/envs/iwod/lib/python3.11/site-packages/lightning/fabric/utilities/cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
/home/dennis/miniconda3/envs/iwod/lib/

Load Dataset

In [5]:
val_dataset = kitti_multiview_detection.KittiMultiviewDataset(dataset_dir=DATASET_PATH,
                                        training="valid_test",
                                        transform=torchvision.transforms.Compose([Normalize(mean_ref=cfg["mean_ref"],
                                                                                std_ref=cfg["std_ref"],
                                                                                means=cfg["mean_target"],
                                                                                stds=cfg["std_target"]),
                                                                                PadImages((cfg["pad_image_h"], cfg["pad_image_w"])),
                                                                                CropImages((cfg["crop_image_h"], cfg["crop_image_w"])),
                                                                                ToTensor()]), cameras=cfg["cameras"],
                                                                                cfg=cfg
                                                                )
dataloader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=16)

Evaluate with all input images

In [6]:
predictions = trainer.predict(model, dataloader, return_predictions=True)
val_img_ids = read_imageset_file(TESTFILE)
kitti_global_detection = get_prediction_dict(predictions, cfg, val_img_ids, score_threshold=0.6)
gt_annos = kitti_eval.get_label_annos(TARGETPATH, val_img_ids)
if os.path.exists(TARGETPATH):
    shutil.rmtree(TARGETPATH)
print(get_official_eval_result(gt_annos, kitti_global_detection, 0))

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
You are using a CUDA device ('NVIDIA A100-SXM4-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]


Output()

/home/dennis/git_repos/IWOD/src/iwod/eval_kitti/utils.py:100: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  nms_idx = dt2.nms_rotated(potential_boxes, torch.tensor(dscores, dtype=potential_boxes.dtype), 0.0001)
/home/dennis/miniconda3/envs/iwod/lib/python3.11/site-packages/numba/cuda/dispatcher.py:536: NumbaPerformanceWarning: Grid size 10 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))
/home/dennis/miniconda3/envs/iwod/lib/python3.11/site-packages/numba/cuda/dispatcher.py:536: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))
/home/dennis/miniconda3/envs/iwod/lib/python3.11/site-packages/numba/cuda/dispatcher.py:536: NumbaPerformanceWarning: Grid size 8 will likely result in GPU under-utilizat

Car AP(Average Precision)@0.70, 0.70, 0.70:
bbox AP:0.00, 0.00, 0.00
bev  AP:45.80, 29.49, 22.92
3d   AP:5.33, 3.61, 2.66
aos  AP:0.00, 0.00, 0.00
Car AP(Average Precision)@0.70, 0.50, 0.50:
bbox AP:0.00, 0.00, 0.00
bev  AP:73.89, 49.27, 40.74
3d   AP:51.75, 34.89, 27.78
aos  AP:0.00, 0.00, 0.00

